# 04 ทดลอง เปรียบเทียบ และประเมิน
ก่อนเริ่ม: ใช้วิธี split และ preprocessing ที่ผ่านการตรวจจาก notebook 03

สมาชิกเขียน baseline และอย่างน้อย 2 อัลกอริทึมเอง เลือกวิธีจากการหารือและหลักฐาน ไม่ตั้งผู้ชนะล่วงหน้า


## เตรียมงานใน Colab
บันทึกสำเนา notebook ใน Drive ก่อน รัน cell นี้เพื่อเตรียม repository ใน runtime เมื่อมีโฟลเดอร์เดิมจะใช้ต่อโดยไม่เขียนทับ สำรองไฟล์ใน Files แยกจาก notebook ก่อนปิด runtime

ใช้ CPU runtime ได้ ไม่ต้องติดตั้งโปรแกรมในเครื่อง


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/Bicycle_Rental_Prediction')]
ROOT = next((p for p in candidates if (p / 'config/project.json').is_file()), None)
if ROOT is None:
    if not Path('/content').is_dir():
        raise RuntimeError('เปิด notebook นี้บน Google Colab และเชื่อม runtime ก่อน')
    ROOT = Path('/content/Bicycle_Rental_Prediction')
    if ROOT.exists():
        raise FileExistsError('พบโฟลเดอร์เดิมที่ยังอ่าน config ไม่ได้ ตรวจและสำรองงานก่อนแก้')
    subprocess.run(['git', 'clone',
                    'https://github.com/Supot77/Bicycle_Rental_Prediction.git',
                    str(ROOT)], check=True)

# Keep Colab's existing scientific stack; install only packages that are missing.
packages = {'pandas': 'pandas', 'sklearn': 'scikit-learn',
            'matplotlib': 'matplotlib', 'joblib': 'joblib'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from starter_support import load_config, configuration_problems, resolve_dataset_path
config = load_config(ROOT)
problems = configuration_problems(config)
if problems:
    raise ValueError('\n'.join(problems))
print('พร้อมทำงานบน Colab: ใช้โฟลเดอร์ repository เดิมถ้ามี โดยไม่ pull/reset ทับงาน')
print('ค่า null ใน config เป็นสถานะรอยืนยัน ไม่ใช่ข้อมูลที่ต้องเดา')


## ทำให้การทดลองทำซ้ำได้
นำโค้ดเตรียมข้อมูลจาก 03 มาเป็นขั้นตอนที่รันได้จากต้น หรือแยกเป็นฟังก์ชันที่นำเข้าได้ตามที่กลุ่มตกลง ห้ามพึ่งตัวแปรที่ค้างในอีก notebook

บันทึก seed ในส่วนที่สุ่ม รุ่นข้อมูล ช่วง split feature และ hyperparameters ใน evidence/experiments.csv


In [ ]:
# เขียนการอ่านข้อมูล split และ preprocessing ที่ทำซ้ำได้ตามข้อตกลงของกลุ่ม


## baseline และ 2 อัลกอริทึม
baseline แบบค่าเฉลี่ยต้องคำนวณจาก train เท่านั้น จากนั้นทดลองอัลกอริทึมที่ต่างกันอย่างน้อยสองแบบ ตัวเลือกและข้อดีข้อเสียอยู่ใน PROJECT_GUIDE_TH

ใช้ validation ชุดเดียวกัน คำนวณ MAE/RMSE/R² และพิจารณาความเร็ว/ขนาด/การอธิบาย baseline เป็นจุดเทียบเพิ่มเติม ไม่ใช่อัลกอริทึมที่สอง


In [ ]:
# เขียน baseline และการฝึกโมเดลที่กลุ่มเลือก พร้อมคำนวณ metrics จริง
# ยังไม่มีผลตัวเลขให้เติมล่วงหน้า


## วิเคราะห์และเลือกภายหลังทดลอง
ดูช่วงที่ error สูง ตัวอย่างที่ผิดมาก และผลต่าง train/validation ตรวจ leakage เมื่อคะแนนดีผิดปกติ R² ไม่ใช่เปอร์เซ็นต์ความแม่น

เลือกแนวทาง/ค่าตั้งจาก validation ก่อน แล้วประเมิน test ที่กันไว้ ถ้าฝึกใหม่ด้วย train+validation ต้องประเมิน artifact ตัวสุดท้ายและคง test แยกไว้

เอกสาร: [Regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)


In [ ]:
# เขียนการประเมินสุดท้ายและการบันทึก preprocessing + model หลังเลือกแล้ว
# บันทึก metadata ให้ตรวจว่าตัวที่โหลดในแอปเป็น artifact รุ่นนี้


## สิ่งส่งต่อและจุดตรวจ
ตารางผลจริงพร้อม experiment IDs เหตุผลเลือก ข้อจำกัด และ artifact ที่โหลดใน session ใหม่ได้ ให้เพื่อนตรวจสูตรและชุดข้อมูลประเมินจากโค้ด


## ก่อนส่งต่อ
เก็บข้อสังเกตและหลักฐานที่เผยแพร่ได้ไว้ใน evidence/results ให้เพื่อนอีกคนตรวจและลองทำซ้ำ จากนั้นล้าง outputs และ execution counts ก่อน commit notebook อย่าใส่ผลสมมติเป็นผลทดลองจริง
